<a href="https://colab.research.google.com/github/Existanze54/sirius-machine-learning-2026/blob/main/Homeworks/HW3_LogReg_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашка 3. Logistic Regression and SVM

### imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression as LogReg
from sklearn.linear_model import LogisticRegressionCV as LogRegCV

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

## Задача 1. Классификация на нуклеотидных k-мерах

**Protein Binding Microarray (PBM)** — один из методов изучения связывания транскрипционных факторов с ДНК. Результаты таких экспериментов позволяют устанавливать мотивы связывания, вроде этого:  
<img src="https://mex.autosome.org/motifs/motif401371.svg" width='300'>

Обычно мотивы объясняют экспериментальные результаты довольно хорошо, но порой алгоритмы ML могут найти более сложные зависимости. В данной задаче предлагается классифицировать олигонуклеотиды на связывающиеся и несвязывающиеся (согласно результатам PBM) с транскрипционным фактором [GCM1](https://www.uniprot.org/uniprotkb/Q9NP62/entry) человека.

In [3]:
import urllib.request

file_id = "1tmbIBuPXhPMjcvAgfs6DFWakOjH6We7O"
url = f"https://docs.google.com/uc?export=download&id={file_id}"

urllib.request.urlretrieve(url, "gcm1.tsv")


('gcm1.tsv', <http.client.HTTPMessage at 0x1807b468e90>)

### Посмотрим на данные и представим их

In [4]:
df = pd.read_csv('gcm1.tsv', sep='\t')
df.head(2)

,#id_spot,row,col,control,id_probe,pbm_sequence,linker_sequence,mean_signal_intensity,mean_background_intensity,flag
0,1,1,1,False,MEforward00001,CGACATGTCGACATCGTCATAAGAGGGATATTCTA,CCTGTGTGAAATTGTTATCCGCTCT,-0.332524,NaN,0
1,171,3,1,False,MEforward00002,GATATTCTAGCACGCCATGTAAATCAAGGTTTACG,CCTGTGTGAAATTGTTATCCGCTCT,-0.721498,NaN,0


`pbm_sequence` это тот самый изучаемый олигонуклеотид, а `mean_signal_intensity` — наша целевая переменная.  
  
До успеха нейросетей, довольно популярным представлением для текстов был **"мешок слов"** — вектор с числом всех встреченных слов. Вам предлагается представлять нуклеотидную последовательность как **"мешок k-меров"**. Для этого я написал несколько функций ниже, можете с ними ознакомиться. Мы будем учитывать все **k-меры** до длины **k = 5** (из-за ограниченности вычислительных ресурсов).

Допишите функции ниже. `get_complement` должна возвращать обратно-комплементарную последовательность, а `get_canonical` — лексикографически меньшую из исходной и обратно-комплементарной строк. Например, для `GT` каноническое представление — `AC`. В `encode_kmers` подсчитайте все перекрывающиеся вхождения фрагментов **каждой длины от 1 до k**. Верните словарь: канонический k-мер → число вхождений. Например, для `AAAA` при `k=2` получится `{"A": 4, "AA": 3}`.

In [33]:
nucleomap = {'A': 'T',
             'C': 'G',
             'G': 'C',
             'T': 'A'}

def get_complement(seq):
    new_seq = ""
    for i in range(len(seq)):
        new_seq += nucleomap[seq[i]]
    return new_seq[::-1]   
        

def get_canonical(kmer):
    return min(kmer, get_complement(kmer))

def encode_kmers(seq, k=5):
    kmer_dict = {}
    for l in range(1, k+1):
        for n in range(len(seq)-l+1):
            kmer = get_canonical(seq[n:n+l])
            kmer_dict.setdefault(kmer, 0)
            kmer_dict[kmer] += 1
    return kmer_dict

In [35]:
encode_kmers("TTTC", k=2)

{'A': 3, 'C': 1, 'AA': 2, 'GA': 1}

In [36]:
seqs = df['pbm_sequence']
kmer_dicts = [encode_kmers(s) for s in seqs]
X = pd.DataFrame(kmer_dicts)
X = X.fillna(0)
X.head(2)

,C,A,CG,GA,AC,CA,AT,TA,AA,AG,...,AATGG,AACAA,CTTGC,AGGAT,ACAAA,TACCA,CCGTC,AAGTC,GAGGA,TGACA
0,15,20,3.0,8.0,4.0,4.0,5.0,3.0,2.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,14,21,2.0,3.0,4.0,4.0,4.0,4.0,6.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Сигнал PBM является вещественной величиной, пропорциональной связыванию белка с олигонуклеотидом. Однако он считается довольно шумным, и связавшимися часто считают только последовательности, давшие сигнал больше 4. Такие мы будем считать положительным классом.

In [37]:
y = df['mean_signal_intensity']
y = y > 4

### Собственно решение

Удалите k-меры, которые встречаются менее чем у 5% олигонуклеотидов. Разделите выборку на обучающую, валидационную и тестовую. Обращаю внимание, что положительного класса у нас всего 1.5%, так что лучше стратифицировать. Стандартизуйте признаки. Обучите логистическую регрессию с L2-регуляризацией. Подберите параметр `C` в диапазоне $[0.01, 100]$ по валидационной выборке. Оцените качество модели с помощью $AUROC$ и $F_1$. Сравните с `LogRegCV` с параметрами по умолчанию.

In [38]:
# your code here

zero_ratio = (X == 0).mean()
cols_to_drop = zero_ratio[zero_ratio > 0.95].index

X = X.drop(columns=cols_to_drop)
X.head(2)



,C,A,CG,GA,AC,CA,AT,TA,AA,AG,...,AATGG,AACAA,CTTGC,AGGAT,ACAAA,TACCA,CCGTC,AAGTC,GAGGA,TGACA
0,15,20,3.0,8.0,4.0,4.0,5.0,3.0,2.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,14,21,2.0,3.0,4.0,4.0,4.0,4.0,6.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [39]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=100,
                                                    stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full,
                                                  test_size=0.3,
                                                  random_state=203,
                                                  stratify=y_train_full)

scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [49]:
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

def train_valid_model(model, X_train, X_val, y_train, y_val):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    auroc = roc_auc_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    return {'F1': f1, 'AUROC': auroc}





In [50]:
c_list = [0.01, 0.1, 1, 10, 100]


scores = []
for c in c_list:
    LR = LogReg(penalty="l2", C=c, max_iter=1000, random_state=42)
    a = train_valid_model(LR, X_train, X_val, y_train, y_val)
    scores.append(a)




C:\AppEnglish\envs\python26\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\AppEnglish\envs\python26\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\AppEnglish\envs\python26\Lib\site-packages\sklearn\

In [51]:
pd.DataFrame(scores, index=c_list)

,F1,AUROC
0.01,0.203390,0.560598
0.10,0.242424,0.580223
1.00,0.238095,0.583051
10.00,0.219298,0.582111
100.00,0.202643,0.575256


## Задача 2. Классификация молекул

Поработаем с известным датасетом [BACE](https://pubmed.ncbi.nlm.nih.gov/27689393/), содержащим молекулы размеченные как ингибиторы и не-ингибиторы человеческой  
*β-секретазы 1*. Дескрипторы (так хемоинформатики называют признаки) рассчитаны за вас.

In [ ]:
! wget -q --no-check-certificate 'https://docs.google.com/uc?export=download&id=1SYdyTbPD01GYcdz7hkyWZGoA5RV44fg7' -O bace.csv

In [ ]:
df = pd.read_csv('bace.csv')
X = df.iloc[:, 4:]
y = df['Class']

Проведите необходимую подготовку данных. Обучите `LogReg` с L1-регуляризацией, а так же `LinearSVC` с L2- и L1-регуляризацией. Подберите параметр `C` для всех трех моделей. Оцените качество моделей метрикой $F_1$.

In [ ]:
# your code here